# Computational Exercises 5

## Exercise 1

The following code implements the projection problem in an arbitrary domain, with any function space and the H1 and L2 inner products

In [34]:
import firedrake as fd
from firedrake.output import VTKFile 
from firedrake.__future__ import interpolate
import numpy as np

def innerL2(u,v):
    return fd.inner(u,v) * fd.dx

def innerH1(u, v):
    return (fd.inner(u,v) + fd.inner(fd.grad(u), fd.grad(v))) * fd.dx


def solveProjection(mesh, u, Vh, inner_type: str):
    uTrial, vh = fd.TrialFunction(Vh), fd.TestFunction(Vh)

    a: ufl.form.Form
    L: ufl.form.Form
    if inner_type == "H1":
        a = innerH1(uTrial, vh)
        L = innerH1(u, vh)
    elif inner_type == "L2":
        a = innerL2(uTrial, vh)
        L = innerL2(u, vh)
    else:
        print("inner type undefined")

    uh = fd.Function(Vh)
    opts={"ksp_type": "preonly", "pc_type": "lu"}
    fd.solve(a == L, uh, solver_parameters=opts)
    
    return uh



## Exercise 2

We're assuming $\Omega = [0,1]$ and $u(x)= sin(2 \pi x)$

In [32]:
lncells = [1, 2, 4, 8, 12, 16, 32]
interval_length = 1.0

#Part 1
lorder = [0, 1, 2]
luhL2 = [[], [], []] #luhL2[i] is the list with results of order i

for i, order in enumerate(lorder):
    for ncells in lncells:
        mesh = fd.IntervalMesh(ncells, interval_length)
        Vh = fd.FunctionSpace(mesh, "DG", degree=order)

        x = fd.SpatialCoordinate(mesh)
        u = fd.sin(2* np.pi * x[0])

        uh = solveProjection(mesh, u, Vh, "L2")
        luhL2[i].append(uh)

#Part 2
lorder = [1, 2, 3]
luhH1 = [[], [], []] #luhH1[i] is the list with results of order i
for i, order in enumerate(lorder):
    for ncells in lncells:
        mesh = fd.IntervalMesh(ncells, interval_length)
        Vh = fd.FunctionSpace(mesh, "CG", degree=order)

        x = fd.SpatialCoordinate(mesh)
        u = fd.sin(2* np.pi * x[0])

        uh = solveProjection(mesh, u, Vh, "H1")
        luhH1[i].append(uh)


#Part 3
print(luhH1[1][1].dat.data[:])

[ 9.51849070e-01  2.99499709e-04  0.00000000e+00 -9.51849070e-01
 -2.99499709e-04]


## Exercise 3

Now $\Omega = [0,1]^2$ and $u(x_1, x_2) = \sin(\pi x_1) \sin(\pi x_2)$

In [37]:
Nx, Ny = 32, 32
Lx, Ly = 1.0, 1.0
mesh = fd.RectangleMesh(Nx, Ny, Lx, Ly, quadrilateral=False)
x = fd.SpatialCoordinate(mesh)

VhL2 = fd.FunctionSpace(mesh, "DG", degree=0)
VhH1 = fd.FunctionSpace(mesh, "CG", degree=1)
u = fd.sin(np.pi * x[0]) * fd.sin( np.pi * x[1])

uhL2 = solveProjection(mesh, u, VhL2, "L2")
uhH1 = solveProjection(mesh, u, VhH1, "H1")

VTKFile("Solutions/uhL2.pvd").write(uhL2)
VTKFile("Solutions/uhH1.pvd").write(uhH1)

We can see the projections:

- $L^2(\Omega)$ projection in Discontinuous Galerkin space

![](imgs/uhL2.png)

- $H^1(\Omega)$ projection in Continuous Galerkin space (order 1)

![](imgs/uhH1.png)